In [13]:
%load_ext autoreload
%autoreload 2


import os
import logging
import pandas as pd
import numpy as np
import time
import torch
import wandb
import copy
from tqdm import tqdm


from src.constants import DEVICE
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler
from src.dataset.load_dataset import load_dataset, generate_K_fold_cross_validation_splits
from src.models.tools.get_classification_model import get_classification_model
from src.dataset.get_dataloader import make_dataloader   
from src.dataset.get_transforms import get_transforms
from src.utils.loss_func.get_loss_function import get_loss_function
from src.evaluation.mainMetricHandler import mainMetricHandler



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import torch

torch.cuda.is_available()


True

In [3]:
from src.config_presets.tools.get_config import get_config

config = get_config('Multi_tox')




src\config_presets\Base_config.yaml
src\config_presets\Multi_tox.yaml


In [4]:
# load the data
df_train_val, df_test = load_dataset(config)
    
dataset_split_dict = generate_K_fold_cross_validation_splits(config, df_train_val)

# cap the number of iterations, if it is less than the number of k-splits to make
n_iterations = config['data']['kFolds']['n_iterations']

# get the data transforms
train_transforms, val_transforms = get_transforms(config)
# get the loss function
loss_function = get_loss_function(config)

metricHandler = mainMetricHandler(config)



Removed patients (no image data) = 0
Train/Val dataset 872 (80.0%), Test dataset 218 (20.0%)


c:\Users\S.P.M. de Vette\.conda\envs\HNC_310\lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [5]:
train_data, val_data = dataset_split_dict[0]['train'], dataset_split_dict[0]['val']

train_loader, metadata = make_dataloader(config, train_data, train_transforms, validation_mode=False)
val_loader, _ = make_dataloader(config, val_data, val_transforms, validation_mode=True)



In [19]:
from src.models.tools.get_classification_model import get_classification_model



config['general']['resultsCurrentDirectory'] = os.getcwd()


config['model']['TransRP']['clinical_features_method'] = 'cls'

model = get_classification_model(config, metadata=metadata, save_summary=True)



''